# Nemotron-3-Nano-30B — v0.14 SFT Training on DGX Spark GB10

**Approach:** Format 4 SFT with per-expert MoE LoRA — 878M trainable params, 11,962 PEFT keys (186 base + 11,776 per-routed-expert)  
**Hardware:** NVIDIA DGX Spark GB10 — 130.7 GB HBM, Blackwell, aarch64  
**Key advance vs v0.12/v0.13:** All **16** competition categories covered — 3 missing categories added via short synthetic CoT traces

For Dockerfile, build instructions, and adapter key taxonomy, see the [v0.9 notebook](https://www.kaggle.com/code/gdataranger/nemotron-v0-9-sft-training-dgx-spark-gb10).  
For v0.12 augmented data and run14/run15, see the [v0.12 notebook](https://www.kaggle.com/code/gdataranger/nemotron-v0-12-sft-training-dgx-spark-gb10).

## Why v0.14 — closing the 3-category gap

v0.12 and v0.13 both miss 3 of 16 competition categories. Every huikang trace for these
categories exceeds the 4,096-token evaluator limit — the token filter drops them all:

| Category | Huikang traces | Median tokens | Status in v0.13 |
|---|---|---|---|
| `spelling` | 1,219 | 4,884 | all dropped by ≤4096 filter |
| `equation_numeric_deduce` | 961 | 5,874 | all dropped by ≤4096 filter |
| `equation_numeric_guess` | 180 | 6,148 | all dropped by ≤4096 filter |

**Solution**: three new short synthetic generators (≤ 500 token CoT) cover each category.

## v0.14 dataset — all 16 categories

**Source**: [gdataranger/nemotron-v014-training-data](https://www.kaggle.com/datasets/gdataranger/nemotron-v014-training-data)

**Build**: 27,500 merged → 17,502 after ≤4096 token filter → 12,800 after cap/repeat

| Category | Count | Notes |
|---|---|---|
| cipher, gravity, matching, numeral, splitting, unit_conversion | 1,500 each | capped |
| equation_symbolic | 500 | synthetic rule-inference puzzles |
| spelling | 500 | **new** synthetic ~150 tok — concat words → `–c–h–a–r–` |
| equation_numeric_deduce | 500 | **new** synthetic ~300 tok — operator → {add, abs_diff} |
| equation_numeric_guess | 500 | **new** synthetic ~350 tok — operator → {add, abs_diff, concat} |
| bit_manipulation | 300 | repeated 10.7× (huikang traces avg 6,971 tok) |
| concatenation, cryptarithm_deduce, cryptarithm_guess, equation_numeric, lstrip | 300 each | repeated |

## Synthetic generators — design

### `scripts/generate_spelling.py` — ~150 token CoT

Rule: concatenate all input words (drop spaces), then surround each character with `–` (EN DASH).

```
Input:  ["hello", "world"]
Output: "–h–e–l–l–o–w–o–r–l–d–"
```

Prompt: 3 sample input→output pairs + 1 test input (Alice's Wonderland format).  
CoT: identify concat+dash rule from examples, apply to test.

### `scripts/generate_equation_numeric_deduce.py` — ~300 token CoT

Each operator symbol maps to one of `{add, abs_diff}`. For unambiguous examples:
- `abs_diff` always uses `a < b` so result = `b−a`, always distinct from `a+b`

CoT: try each operation per symbol against all example pairs, declare winner.

### `scripts/generate_equation_numeric_guess.py` — ~350 token CoT

Operations: `{add, abs_diff, concat}`. Always one concat symbol + one numeric symbol per puzzle.
- `concat` example: `39 @ 28 = 3928` — always 4-digit, unambiguous vs add (≤198) and abs_diff (≤88)

### Format 4 compliance

All four generators produce **Format 4** — no `<think>` opener:

```
<|im_start|>assistant
{reasoning trace}
</think>
\boxed{answer}<|im_end|>
```

## Run history

| Run | Script | Steps | Seq | Warmstart | LR | Kaggle Score | Notes |
|---|---|---|---|---|---|---|---|
| run16 | `run_train_v13.sh` | **17 (stopped)** | 4096 | run15 step-200 | 1e-4 | — | Pivoted to v0.14 (v0.13 missing 3 cats) |
| **run17** | `run_train_v14.sh` | 500 planned | 4096 | run15 step-200 | 1e-4 | pending | All 16 cats, 12,800 rows — **active** |

In [ ]:
# ── RUN IDENTITY ──────────────────────────────────────────────────────────
RUN_NAME = "v14_run17"

# ── DATA ─────────────────────────────────────────────────────────────────
TRAIN_FILE     = "data/v0.14_train.jsonl"   # 12,800 rows, all 16 categories
MAX_SEQ_LENGTH = 4096

# ── INCREMENTAL TRAINING ──────────────────────────────────────────────────
WARMSTART_ADAPTER = "output/adapter_v12_run15_step200"

# ── TRAINING HYPERPARAMETERS ─────────────────────────────────────────────
MAX_STEPS     = 500       # ~15.6h at 107 s/step
LEARNING_RATE = 1e-4
BATCH_SIZE    = 1
GRAD_ACCUM    = 16        # effective batch = 16
LORA_R        = 32
LORA_ALPHA    = 32
CKPT_EVERY    = 100
SEED          = 3407

print(f"RUN_NAME:          {RUN_NAME}")
print(f"TRAIN_FILE:        {TRAIN_FILE}")
print(f"WARMSTART_ADAPTER: {WARMSTART_ADAPTER}")
print(f"MAX_SEQ_LENGTH:    {MAX_SEQ_LENGTH}")
print(f"MAX_STEPS:         {MAX_STEPS}")
print(f"LEARNING_RATE:     {LEARNING_RATE}")
print(f"CKPT_EVERY:        {CKPT_EVERY}")
print()
print("NOTE: Training runs on the DGX Spark GB10 host.")
print("      This notebook is a documentation reference — cells below are read-only on Kaggle.")

## Build the v0.14 dataset

`scripts/generate_v14_data.sh` runs all 4 generators, merges with v0.12 base, then applies
token-filter + cap/repeat inside Docker.

```zsh
# From project root on DGX Spark host (~2 min total)
bash scripts/generate_v14_data.sh
```

Or step by step:

```zsh
# Step 1: generate synthetic examples (no Docker, ~seconds each)
python3 scripts/generate_spelling.py               --n 500 --seed 42 --out data/spelling_synthetic.jsonl
python3 scripts/generate_equation_numeric_deduce.py --n 500 --seed 42 --out data/eq_num_deduce_synthetic.jsonl
python3 scripts/generate_equation_numeric_guess.py  --n 500 --seed 42 --out data/eq_num_guess_synthetic.jsonl
python3 scripts/generate_equation_symbolic.py       --n 500 --seed 42 --out data/equation_symbolic_synthetic.jsonl

# Step 2: merge with v0.12 base (27,500 rows)
cat data/v0.12_train.jsonl \
    data/equation_symbolic_synthetic.jsonl \
    data/spelling_synthetic.jsonl \
    data/eq_num_deduce_synthetic.jsonl \
    data/eq_num_guess_synthetic.jsonl \
    > data/v0.14_merged.jsonl

# Step 3: token-filter (≤4096 tok) + cap/repeat → 12,800 final (requires Docker)
docker run --rm --privileged -e NVIDIA_VISIBLE_DEVICES=all \
  -e HF_HUB_OFFLINE=1 -e TRANSFORMERS_OFFLINE=1 \
  --user $(id -u):$(id -g) \
  -v $(pwd):/workspace \
  -v $(pwd)/.cache/huggingface:/home/ubuntu/.cache/huggingface \
  -w /workspace nemotron-gb10:latest \
  python3 scripts/balance_dataset.py \
    --input  data/v0.14_merged.jsonl \
    --output data/v0.14_train.jsonl \
    --max-tokens 4096 \
    --max-per-category 1500 \
    --min-per-category 300 \
    --seed 42
```

**Expected output:**
```
Kept 12,800 / dropped 0
stratified over 16 categories
```

## Training — v0.14 run17 (`run_train_v14.sh`)

```zsh
# Always in tmux — never run directly
tmux new -s train_v14

WARMSTART_ADAPTER=output/adapter_v12_run15_step200 \
RUN_NAME=v14_run17 \
bash scripts/run_train_v14.sh
```

**Startup confirmations** (verify before stepping away):
```
[moe-lora] Warmstart: loaded 92 expert LoRA weights from output/adapter_v12_run15_step200/expert_lora_weights.pt
Kept 12,800 / dropped 0
stratified over 16 categories
Trainable: 878,880,768 / 32,456,818,112 (2.71%)
```

**Step time:** ~107 s/step → 500 steps ≈ 15h.

In [ ]:
import re, pathlib, subprocess

log_path = pathlib.Path(f"output/train_{RUN_NAME}.log")

if not log_path.exists():
    print(f"Log not found: {log_path}")
    print("Training runs on the DGX Spark GB10 host — this cell reads the live log there.")
    print("On Kaggle: not applicable.")
else:
    result = subprocess.run(["tail", "-20", str(log_path)], capture_output=True, text=True)
    print(result.stdout)

    # Parse step directly from tqdm bar: "| 50/500 [...] {'loss': '0.2632', ...}"
    loss_pat = re.compile(r"\|\s+(\d+)/\d+\s+\[.*?\].*?'loss':\s+'([0-9.]+)'.*?'learning_rate':\s+'([0-9e.+-]+)'")
    ckpt_pat = re.compile(r"\[ckpt\] step (\d+)")
    losses, checkpoints = [], []
    with open(log_path) as f:
        for line in f:
            m = loss_pat.search(line)
            if m:
                losses.append((int(m.group(1)), float(m.group(2)), m.group(3)))
            c = ckpt_pat.search(line)
            if c:
                checkpoints.append(int(c.group(1)))

    if losses:
        print(f"\nLoss trajectory ({len(losses)} readings):")
        print(f"  {'step':>5}  {'loss':>7}  {'lr':>10}")
        for step, loss, lr in losses[-15:]:
            print(f"  {step:>5}  {loss:>7.4f}  {lr:>10}")
    if checkpoints:
        print(f"\nCheckpoints saved at steps: {checkpoints}")
        print(f"Rolling checkpoint: output/adapter_{RUN_NAME}_ckpt/")

## Package and submit

Package from the **rolling checkpoint** immediately after each 100-step notification.

```zsh
# Package from rolling checkpoint (run on HOST, not inside Docker)
bash scripts/package_submission.sh \
  output/adapter_v14_run17_ckpt \
  /tmp/sub_v14_step<N>

kaggle competitions submit \
  -c nvidia-nemotron-model-reasoning-challenge \
  -f /tmp/sub_v14_step<N>/submission.zip \
  -m "v0.14 run17 step-<N>: all 16 cats, warmstart run15-step200"
```

Submit at steps 100, 200, 300, 400, 500 and compare Kaggle scores.

## Submission verification

A valid submission zip contains:
- `adapter_config.json` — PEFT config with Kaggle model path (fixed by `package_submission.sh`)
- `adapter_model.safetensors` — 11,962 LoRA keys (186 base + 11,776 per-expert)

Expert LoRA key form: `base_model.model.model.layers.{i}.mixer.experts.{j}.up_proj.lora_A.default.weight`  
23 layers × 128 experts × 2 proj × 2 (A/B) = **11,776 keys** — all loaded at Kaggle inference.